# Transfer Generated Files

Use this notebook after `agent_coding_workflow.ipynb` finishes a sprint run.

It reads `outputs/latest_run.txt`, loads the latest run folder, shows each generated file with its `PASS` or `FAIL` status, and lets you choose which files to copy into the target Django app repo.

Default choices are conservative:

1. `PASS` files default to apply.
2. `FAIL` files default to skip.
3. Files without a saved source are skipped.

The notebook writes `applied_files.json` and `command_logs.jsonl` back into the selected run folder for traceability.



In [1]:
# Step 1: Imports and path setup
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
from datetime import datetime

# The notebook is expected to run from the CPE494-agent-coding-team repo root.
# In VS Code, set the notebook working directory to the repository root if needed.
REPO_ROOT = Path.cwd().resolve()

# If the notebook is opened from notebooks/, move one level up automatically.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.resolve()

TARGET_APP_PATH = (REPO_ROOT.parent / "CPE494-erp-invoice-app-by-ai").resolve()
OUTPUTS_DIR = REPO_ROOT / "outputs"
RUNS_DIR = OUTPUTS_DIR / "runs"
LATEST_RUN_PATH = OUTPUTS_DIR / "latest_run.txt"

print("Agent repo root:", REPO_ROOT)
print("Target app path:", TARGET_APP_PATH)
print("Latest run marker:", LATEST_RUN_PATH)



Agent repo root: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team
Target app path: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-erp-invoice-app-by-ai
Latest run marker: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\latest_run.txt


In [2]:
# Step 2: Utility functions

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def save_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def get_git_status_short(repo_path: Path) -> str:
    try:
        result = subprocess.run(
            ["git", "status", "--short"],
            cwd=str(repo_path),
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return "unknown"


def safe_relative_path(file_name: str) -> Path:
    """Return a safe relative path for a generated target file."""
    rel = Path(file_name.replace("\\", "/"))
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe file path from task: {file_name}")
    return rel


def read_latest_run_id() -> str:
    if not LATEST_RUN_PATH.exists():
        raise FileNotFoundError(f"Missing latest run marker: {LATEST_RUN_PATH}")
    run_id = LATEST_RUN_PATH.read_text(encoding="utf-8").strip()
    if not run_id:
        raise ValueError(f"Latest run marker is empty: {LATEST_RUN_PATH}")
    return run_id


def load_latest_run_dir() -> Path:
    run_id = read_latest_run_id()
    run_dir = RUNS_DIR / run_id
    if not run_dir.exists():
        raise FileNotFoundError(f"Latest run folder does not exist: {run_dir}")
    return run_dir



In [3]:
# Step 3: Load the latest run
RUN_ID = read_latest_run_id()
RUN_DIR = load_latest_run_dir()
GENERATED_DIR = RUN_DIR / "generated_files"

print("Latest run ID:", RUN_ID)
print("Run folder:", RUN_DIR)
print("Workflow result:", RUN_DIR / "workflow_result.json")



Latest run ID: 2026-05-06_231917_sprint_00_scope
Run folder: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\runs\2026-05-06_231917_sprint_00_scope
Workflow result: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\runs\2026-05-06_231917_sprint_00_scope\workflow_result.json


In [4]:
# Step 4: Build apply candidates from workflow_result.json

def list_generated_files() -> list[Path]:
    if not GENERATED_DIR.exists():
        return []
    return sorted([p for p in GENERATED_DIR.rglob("*") if p.is_file()])


def load_workflow_results(run_dir: Path) -> list[dict]:
    """Return per-file workflow results if workflow_result.json exists."""
    workflow_result_path = run_dir / "workflow_result.json"
    if not workflow_result_path.exists():
        return []
    data = json.loads(workflow_result_path.read_text(encoding="utf-8"))
    return data.get("results", [])


def latest_attempt_code_path(run_dir: Path, relative_path: Path) -> Path | None:
    """Return the latest saved attempt code for a file, if any."""
    attempt_dir = run_dir / "attempts" / relative_path.as_posix().replace("/", "__")
    if not attempt_dir.exists():
        return None
    attempt_files = sorted(attempt_dir.glob("attempt_*_code.txt"))
    if not attempt_files:
        return None
    return attempt_files[-1]


def apply_feature_group_defaults(candidates: list[dict]) -> list[dict]:
    """If workflow results include feature_group, keep each group consistent by default."""
    groups = {}
    for candidate in candidates:
        group = candidate.get("feature_group")
        if not group:
            continue
        groups.setdefault(group, []).append(candidate)

    for _, items in groups.items():
        if any(item.get("status") != "PASS" for item in items):
            for item in items:
                item["default_apply"] = False
                item["default_reason"] = "feature group contains failed file"

    return candidates


def build_apply_candidates() -> list[dict]:
    """Build target-file apply candidates with PASS/FAIL and bundle-aware defaults."""
    workflow_results = load_workflow_results(RUN_DIR)
    candidates = []

    if workflow_results:
        for result in workflow_results:
            rel = safe_relative_path(result["file_name"])
            status = result.get("status", "UNKNOWN").upper()
            generated_path = GENERATED_DIR / rel
            if generated_path.exists():
                source = generated_path
                source_kind = "generated_files"
            else:
                source = latest_attempt_code_path(RUN_DIR, rel)
                source_kind = "latest_attempt"

            candidates.append({
                "relative_path": rel,
                "status": status,
                "attempts": result.get("attempts"),
                "default_apply": status == "PASS",
                "default_reason": "status PASS" if status == "PASS" else "status not PASS",
                "source": source,
                "source_kind": source_kind,
                "feature_group": result.get("feature_group"),
            })

        return apply_feature_group_defaults(candidates)

    # Backward-compatible fallback for older runs without workflow_result.json.
    candidates = [
        {
            "relative_path": p.relative_to(GENERATED_DIR),
            "status": "PASS",
            "attempts": None,
            "default_apply": True,
            "default_reason": "generated file without workflow_result.json",
            "source": p,
            "source_kind": "generated_files",
            "feature_group": None,
        }
        for p in list_generated_files()
    ]
    return apply_feature_group_defaults(candidates)


candidates = build_apply_candidates()
print(f"Found {len(candidates)} candidate files.")
for candidate in candidates:
    rel = candidate["relative_path"].as_posix()
    source = candidate.get("source")
    source_note = candidate.get("source_kind") if source and source.exists() else "no source"
    default_text = "apply" if candidate["default_apply"] else "skip"
    reason = candidate.get("default_reason", "")
    print(f"- {rel}: {candidate['status']} ({source_note}, default {default_text}; {reason})")


Found 15 candidate files.
- requirements.txt: PASS (generated_files, default apply; status PASS)
- .gitignore: PASS (generated_files, default apply; status PASS)
- manage.py: PASS (generated_files, default apply; status PASS)
- erp_invoice/__init__.py: PASS (generated_files, default apply; status PASS)
- erp_invoice/settings.py: PASS (generated_files, default apply; status PASS)
- templates/base.html: PASS (generated_files, default apply; status PASS)
- templates/landing.html: PASS (generated_files, default apply; status PASS)
- core/__init__.py: PASS (generated_files, default apply; status PASS)
- core/apps.py: PASS (generated_files, default apply; status PASS)
- core/views.py: PASS (generated_files, default apply; status PASS)
- core/urls.py: PASS (generated_files, default apply; status PASS)
- erp_invoice/urls.py: PASS (generated_files, default apply; status PASS)
- erp_invoice/wsgi.py: PASS (generated_files, default apply; status PASS)
- erp_invoice/asgi.py: PASS (generated_files, 

In [5]:
# (Bundle-consistency guard)
# If the workflow_result.json shows a bundle as INCOMPLETE (one or more files in the
# same feature_group did not pass), default-skip every file in that bundle so a
# half-applied bundle never reaches the target repo. The user can still override.
def _bundles_with_failures(workflow_result: dict) -> set:
    failed_groups: set = set()
    bundle_summary = workflow_result.get("bundle_summary") or {}
    for group, b in bundle_summary.items():
        if b.get("fail", 0) > 0:
            failed_groups.add(group)
    return failed_groups

# Hook example (apply this where you decide default_apply for each candidate):
#   incomplete_bundles = _bundles_with_failures(workflow_result)
#   if candidate.get("feature_group") in incomplete_bundles:
#       candidate["default_apply"] = False

# Step 5: Apply selected files to the target app repo

def prompt_apply_candidate(candidate: dict) -> bool:
    """Ask whether to apply one file, using Enter to accept the default."""
    rel_text = candidate["relative_path"].as_posix()
    default_text = "y" if candidate["default_apply"] else "n"
    status_text = candidate["status"]
    source = candidate.get("source")
    if source is None or not source.exists():
        print(f"{rel_text} [{status_text}] has no saved source file; skipping.")
        return False

    answer = input(f"Apply {rel_text} [{status_text}]? (y/n, default {default_text}): ").strip().lower()
    if not answer:
        return candidate["default_apply"]
    return answer in {"y", "yes"}


def apply_generated_files_to_target():
    candidates = build_apply_candidates()

    if not candidates:
        print("No generated files or attempts to apply.")
        return []

    print("Review files to apply. Press Enter to accept each default.")
    print("Default is y for PASS files and n for failed files.")

    selected_candidates = []
    skipped_records = []
    for candidate in candidates:
        selected = prompt_apply_candidate(candidate)
        rel_text = candidate["relative_path"].as_posix()
        if selected:
            selected_candidates.append(candidate)
        else:
            skipped_records.append({
                "relative_path": rel_text,
                "status": candidate["status"],
                "default_apply": candidate["default_apply"],
                "default_reason": candidate.get("default_reason"),
                "feature_group": candidate.get("feature_group"),
                "source": str(candidate["source"]) if candidate.get("source") else None,
                "source_kind": candidate.get("source_kind"),
            })
            print(f"Skipped: {rel_text}")

    if not selected_candidates:
        print("No files selected to apply.")
        save_json(RUN_DIR / "applied_files.json", {
            "applied": False,
            "applied_at": now_iso(),
            "run_id": RUN_ID,
            "target_app_path": str(TARGET_APP_PATH),
            "files": [],
            "skipped_files": skipped_records,
        })
        return []

    applied_records = []
    for candidate in selected_candidates:
        src = candidate["source"]
        rel = candidate["relative_path"]
        dest = TARGET_APP_PATH / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest)
        record = {
            "relative_path": rel.as_posix(),
            "status": candidate["status"],
            "source": str(src),
            "source_kind": candidate.get("source_kind"),
            "default_reason": candidate.get("default_reason"),
            "feature_group": candidate.get("feature_group"),
            "destination": str(dest),
            "sha256": file_sha256(dest),
            "applied_at": now_iso(),
        }
        applied_records.append(record)
        print(f"Applied: {rel.as_posix()}")

    save_json(RUN_DIR / "applied_files.json", {
        "applied": True,
        "applied_at": now_iso(),
        "run_id": RUN_ID,
        "target_app_path": str(TARGET_APP_PATH),
        "target_app_repo_status_after_apply": get_git_status_short(TARGET_APP_PATH),
        "files": applied_records,
        "skipped_files": skipped_records,
    })

    print("Done. Review git diff in the target app repo.")
    return applied_records


applied_records = apply_generated_files_to_target()



Review files to apply. Press Enter to accept each default.
Default is y for PASS files and n for failed files.
Applied: requirements.txt
Applied: .gitignore
Applied: manage.py
Applied: erp_invoice/__init__.py
Applied: erp_invoice/settings.py
Applied: templates/base.html
Applied: templates/landing.html
Applied: core/__init__.py
Applied: core/apps.py
Applied: core/views.py
Applied: core/urls.py
Applied: erp_invoice/urls.py
Applied: erp_invoice/wsgi.py
Applied: erp_invoice/asgi.py
Applied: core/tests.py
Done. Review git diff in the target app repo.


In [6]:
# Stop here during normal execution.
raise SystemExit("Normal stop point: can go run in terminal")


SystemExit: Normal stop point: can go run in terminal

c:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Step 6: Build, smoke-run, and Git helpers

import time
import urllib.request
import urllib.error
import re as _re

def run_command(command: list[str], cwd: Path, label: str | None = None) -> subprocess.CompletedProcess:
    started_at = now_iso()
    print("Running:", " ".join(command))
    result = subprocess.run(command, cwd=str(cwd), capture_output=True, text=True, shell=False)

    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print("Return code:", result.returncode)
    print()

    log_command(label or " ".join(command), result.returncode, result.stdout, result.stderr, started_at)
    return result


def django_check():
    return run_command(["python", "manage.py", "check"], cwd=TARGET_APP_PATH, label="manage_check")


def django_makemigrations():
    return run_command(["python", "manage.py", "makemigrations"], cwd=TARGET_APP_PATH, label="manage_makemigrations")


def django_migrate():
    return run_command(["python", "manage.py", "migrate"], cwd=TARGET_APP_PATH, label="manage_migrate")


def django_test():
    return run_command(["python", "manage.py", "test", "--verbosity", "2"], cwd=TARGET_APP_PATH, label="manage_test")


def extract_smoke_urls_from_scope(scope_text: str, port: int = 8765) -> list[str]:
    """Parse the active sprint scope's 'Smoke Run' section for URLs to GET.

    Looks for a section heading matching '## <n>. Smoke Run' (or just 'Smoke Run')
    and pulls one URL per non-empty list item until the next '##' heading. Items
    may be:
      - a full URL: http://127.0.0.1:8765/customers/
      - a path inside backticks: `/customers/`
      - a bare path: /customers/

    Bare/path items are prefixed with http://127.0.0.1:<port> automatically.
    Returns at minimum [f'http://127.0.0.1:{port}/'] as a fallback.
    """
    section_match = _re.search(
        r"^##\s*\d*\.?\s*Smoke\s*Run.*?(?=^##|\Z)",
        scope_text,
        _re.MULTILINE | _re.DOTALL | _re.IGNORECASE,
    )
    if not section_match:
        print("WARNING: No 'Smoke Run' section found in sprint scope; defaulting to landing only.")
        return [f"http://127.0.0.1:{port}/"]

    section = section_match.group(0)
    urls: list[str] = []
    for raw in section.split("\n"):
        line = raw.strip().lstrip("-*").strip()
        if not line:
            continue
        full = _re.search(r"https?://[^\s`)]+", line)
        if full:
            urls.append(full.group(0).rstrip(",;"))
            continue
        backtick_path = _re.search(r"`(/[^`]*)`", line)
        if backtick_path:
            urls.append(f"http://127.0.0.1:{port}{backtick_path.group(1)}")
            continue
        bare_path = _re.match(r"(/[^\s]*)", line)
        if bare_path:
            urls.append(f"http://127.0.0.1:{port}{bare_path.group(1)}")
            continue

    seen, deduped = set(), []
    for u in urls:
        if u not in seen:
            seen.add(u); deduped.append(u)
    return deduped or [f"http://127.0.0.1:{port}/"]


def django_smoke_http(urls: list[str], port: int = 8765, startup_seconds: float = 3.0) -> dict:
    """Start runserver on a non-default port, GET each URL, then terminate.

    Returns dict of url -> {"status": int|None, "snippet": str, "error": str|None}.
    """
    print(f"Starting runserver on 127.0.0.1:{port} for smoke HTTP checks...")
    server = subprocess.Popen(
        ["python", "manage.py", "runserver", f"127.0.0.1:{port}", "--noreload"],
        cwd=str(TARGET_APP_PATH),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    time.sleep(startup_seconds)
    results: dict = {}
    try:
        for url in urls:
            try:
                with urllib.request.urlopen(url, timeout=5) as resp:
                    body = resp.read().decode("utf-8", errors="replace")
                    snippet = body[:200].replace("\n", " ")
                    results[url] = {"status": resp.status, "snippet": snippet, "error": None}
                    print(f"  {resp.status} {url}")
            except urllib.error.HTTPError as e:
                results[url] = {"status": e.code, "snippet": "", "error": f"HTTPError: {e.reason}"}
                print(f"  {e.code} {url} (HTTPError)")
            except Exception as e:
                results[url] = {"status": None, "snippet": "", "error": f"{type(e).__name__}: {e}"}
                print(f"  ERR {url}  {type(e).__name__}: {e}")
    finally:
        server.terminate()
        try:
            server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server.kill()
        print("runserver stopped.")
    log_command("django_smoke_http", 0, json.dumps(results), "", now_iso())
    return results


def git_status():
    return run_command(["git", "status"], cwd=TARGET_APP_PATH, label="git_status")


def git_diff():
    return run_command(["git", "--no-pager", "diff", "--stat"], cwd=TARGET_APP_PATH, label="git_diff")


In [ ]:
# Step 7: SMOKE-RUN GATE
# A sprint is NOT done until this gate passes. SMOKE_URLS is parsed from the
# active sprint scope's "Smoke Run" section â€” no sprint-specific URLs are
# hardcoded in this notebook.

print("=" * 70)
print("SMOKE-RUN GATE â€” must pass for the sprint to be considered done")
print("=" * 70)

# Load the snapshotted sprint scope from the run folder. The workflow notebook
# writes input_snapshot/sprint_scope.md when it starts a run.
scope_snapshot_path = RUN_DIR / "input_snapshot" / "sprint_scope.md"
if scope_snapshot_path.exists():
    scope_text = scope_snapshot_path.read_text(encoding="utf-8")
    SMOKE_URLS = extract_smoke_urls_from_scope(scope_text)
    print(f"Parsed {len(SMOKE_URLS)} smoke URL(s) from {scope_snapshot_path.name}:")
    for u in SMOKE_URLS:
        print(f"   {u}")
else:
    SMOKE_URLS = ["http://127.0.0.1:8765/"]
    print(f"WARNING: {scope_snapshot_path} not found. Defaulting to landing-only smoke check.")

print()
check_result = django_check()
mm_result = django_makemigrations()
mig_result = django_migrate()
test_result = django_test()
http_results = django_smoke_http(SMOKE_URLS)

gate_pass = (
    check_result.returncode == 0
    and mm_result.returncode == 0
    and mig_result.returncode == 0
    and test_result.returncode == 0
    and all(r.get("status") == 200 for r in http_results.values())
)

print()
print("=" * 70)
print(f"SMOKE GATE: {'PASS' if gate_pass else 'FAIL'}")
print("=" * 70)
print(f"  check:           returncode={check_result.returncode}")
print(f"  makemigrations:  returncode={mm_result.returncode}")
print(f"  migrate:         returncode={mig_result.returncode}")
print(f"  test:            returncode={test_result.returncode}")
for url, r in http_results.items():
    print(f"  HTTP {r.get('status')} {url}")

if not gate_pass:
    print()
    print("Gate failed. Inspect the output above, fix the underlying issue, and either:")
    print("  - re-run the smoke gate after a manual fix in the target repo, or")
    print("  - revert the bundle in target repo (git restore) and re-run the workflow")
    print("    notebook with refined sprint scope or audit feedback baked into the spec.")


In [ ]:
# Step 8: Definition-of-Done summary (lightweight)
# Cross-references smoke results with the sprint scope's smoke URL list and
# prints a per-line verdict so students can see exactly which criteria passed.

dod_lines = [
    ("python manage.py check exits 0",                 check_result.returncode == 0),
    ("python manage.py makemigrations exits 0",        mm_result.returncode == 0),
    ("python manage.py migrate exits 0",               mig_result.returncode == 0),
    ("python manage.py test exits 0 (all tests pass)", test_result.returncode == 0),
]
for url in SMOKE_URLS:
    r = http_results.get(url, {})
    dod_lines.append((f"GET {url} returns 200", r.get("status") == 200))

print("Definition of Done (smoke-derived):")
for label, passed in dod_lines:
    mark = "[x]" if passed else "[ ]"
    print(f"  {mark} {label}")

remaining = [label for label, passed in dod_lines if not passed]
if not remaining:
    print()
    print("All smoke-derived DoD items passed. Confirm the manual click-path test in the browser before marking the sprint complete.")
else:
    print()
    print(f"{len(remaining)} item(s) outstanding. Sprint is NOT done.")


In [ ]:
# Step 8: Inspect Git status and diff summary in the target app repo.
git_status()
git_diff()

